### Notebook
- **Name:** `cs_rename_files.ipynb`
- **Created/updated:** 2026-02-27
- **Python:** 3.x

### Purpose
Normalize/rename WindNinja output files to the project naming convention.

### Inputs
WindNinja output directory with raw result files.

### Outputs
Renamed files in the WindNinja results directory.

### Dependencies
- (see imports below)

### Usage
Executed by the project pipeline (e.g., via Papermill) or run interactively in Jupyter.

### Notes
- Keep paths and parameters centralized in `config.toml` / `CONFIG_PATH` where applicable.


In [ ]:
# Parameters (Papermill)
CONFIG_PATH =  "" # r"E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\config.toml" # "config.toml"
PROJECT_ROOT = r"E:\mario\python\criticalspam"

### Rename WindNinja results


### Packages


In [10]:

from pathlib import Path
from typing import Optional

import tomllib

import time, subprocess

from dataclasses import dataclass


from __future__ import annotations

import re
import pandas as pd

import sys  

In [11]:
# Ruta al directorio raíz del proyecto (sube 1 nivel desde notebooks/)
#ROOT = Path.cwd().resolve().parent

# Añade src/ al path de importación
SRC = Path(PROJECT_ROOT + r"\src") 
print(f"Adding to sys.path: {SRC}")

#SRC =r'E:\mario\python\criticalspam\src'

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Ya puedes importar
from cs_utils import Config, join_base, load_config_toml

Adding to sys.path: E:\mario\python\criticalspam\src


In [12]:
#CONFIG_PATH = "E:\\mario\\trabajos2\\viesgo_edp_portugal_cic\\estudios_microclimaticos\\Corredoria_Grado_1_y_2\\config.toml"

cfg = load_config_toml(CONFIG_PATH)

print("cs_rename_files.ipynb")
print("Config path:", CONFIG_PATH)

cs_rename_files.ipynb
Config path: E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\config.toml


In [13]:

# ============================================================
# PARÁMETROS (EDITA AQUÍ)
# ============================================================
CASES_CSV = cfg.out_weather_point_file # Path(r"/ruta/a/WN_input_Point_1.csv")     # <-- fichero de casos
OUT_DIR   = cfg.out_wn # Path(r"/ruta/a/salidas_windninja")       # <-- carpeta con resultados WindNinja
DEST_DIR = cfg.out_wn_ren # Path(r"/ruta/a/salidas_windninja_renombrados")  # <-- carpeta destino renombrados

DIAG_CSV = Path(join_base(cfg.general_path, Path("rename_diagnostics.csv")))
SUMMARY_TXT = Path(join_base(cfg.general_path, Path("rename_summary.txt")))



PREFIX = cfg.line # "WN_Corredoria_Grado_point_01"              # <-- regla 2

DATE_COL  = "date_time"                               # <-- columna en el CSV
DIR_COL   = "Direction(degrees)"                      # <-- columna en el CSV
SPEED_COL = "Speed"                                   # <-- columna en el CSV

RECURSIVE = False                                     # True si hay subcarpetas
APPLY     = True                                     # True para renombrar físicamente / False solo planificar para ver asociaciones creadas


plan_csv_path = join_base( cfg.general_path, Path("rename_plan.csv"))
PLAN_CSV  = Path( plan_csv_path)                   # salida del plan old->new

# Si tus ficheros terminan exactamente en _vel.asc / _vel.prj (sin 100m_):
SUFFIX_RE = re.compile(r"(_(?:ang|cld|vel)\.(?:asc|prj))$", re.IGNORECASE)

# Si en realidad terminan como _100m_vel.asc / _100m_vel.prj y quieres conservar "100m_",
# comenta la línea anterior y usa esta:
# SUFFIX_RE = re.compile(r"(_[^_]*?(?:ang|cld|vel)\.(?:asc|prj))$", re.IGNORECASE)

# ============================================================
# IMPLEMENTACIÓN
# ============================================================
EXPECTED_SUFFIXES = {
    "_ang.asc", "_ang.prj",
    "_cld.asc", "_cld.prj",
    "_vel.asc", "_vel.prj",
}


# Si necesitas conservar también un prefijo tipo "100m_" (p.ej. _100m_vel.asc), usa:
# SUFFIX_RE = re.compile(r"(_[^_]*?(?:ang|cld|vel)\.(?:asc|prj))$", re.IGNORECASE)


# ============================================================
# FUNCIONES DE FORMATEO
# ============================================================
def fmt_datetime(dt_str: str) -> str:
    dt = pd.to_datetime(dt_str, utc=True, errors="raise")
    return dt.strftime("%m-%d-%Y_%H%M")   # <-- CORRECTO


def fmt_dir(deg: float) -> str:
    s = f"{deg:.1f}"
    int_part, dec_part = s.split(".")
    return f"{int(int_part):03d}_{dec_part}"


def fmt_speed(spd: float) -> str:
    s = f"{spd:.1f}"
    int_part, dec_part = s.split(".")
    return f"{int(int_part):02d}_{dec_part}"


# ============================================================
# LISTADO DE FICHEROS (SOLO NOMBRES)
# ============================================================
def list_candidate_files(out_dir: Path, recursive: bool) -> dict[str, Path]:
    """
    Devuelve un mapa:
      nombre_de_fichero -> Path completo
    Solo para ficheros que cumplen SUFFIX_RE.
    """
    if recursive:
        it = out_dir.rglob("*")
    else:
        it = out_dir.glob("*")

    name_to_path: dict[str, Path] = {}
    for p in it:
        if p.is_file() and SUFFIX_RE.search(p.name):
            # Nota: si hubiera colisiones de nombre en subcarpetas distintas,
            # la última sobrescribiría. Si ese es tu caso, conviene guardar relative_path.
            name_to_path[p.name] = p
    return name_to_path


# ============================================================
# PLAN DE RENOMBRADO (SOLO NOMBRES)
# ============================================================
def build_plan_names_only(
    cases_csv: Path,
    out_dir: Path,
    prefix: str,
    date_col: str,
    dir_col: str,
    speed_col: str,
    recursive: bool,
) -> tuple[pd.DataFrame, dict]:
    if not cases_csv.exists():
        raise FileNotFoundError(f"No existe CASES_CSV: {cases_csv}")
    if not out_dir.exists() or not out_dir.is_dir():
        raise NotADirectoryError(f"No existe OUT_DIR o no es un directorio: {out_dir}")

    df = pd.read_csv(cases_csv)
    for c in (date_col, dir_col, speed_col):
        if c not in df.columns:
            raise KeyError(f"Columna '{c}' no encontrada en {cases_csv}. Columnas: {list(df.columns)}")

    df["dt_key"]  = df[date_col].map(fmt_datetime)
    df["dir_key"] = df[dir_col].map(fmt_dir)
    df["spd_key"] = df[speed_col].map(fmt_speed)

    df["target_base"] = (
        prefix + "_" +
        df["dir_key"] + "_" +
        df["spd_key"] + "_" +
        df["dt_key"]
    )

    name_to_path = list_candidate_files(out_dir, recursive=recursive)
    candidate_names = list(name_to_path.keys())

    plan_rows = []
    missing_cases = []
    incomplete_cases = []

    for _, row in df.iterrows():
        dt_key = row["dt_key"]
        base   = row["target_base"]

        # Candidatos cuyo nombre contiene dt_key
        # matches = [n for n in candidate_names if dt_key in n]
        
        matches = []
        for n in candidate_names:
            m = re.search(r"\d{2}-\d{2}-\d{4}_\d{4}", n)
            if m and m.group(0) == dt_key:
                matches.append(n)
        
        
        
        
        
        if not matches:
            missing_cases.append(dt_key)
            continue

        # extrae sufijos distintos encontrados
        found: dict[str, str] = {}  # suffix -> old_name
        for old_name in matches:
            m = SUFFIX_RE.search(old_name)
            if not m:
                continue
            suffix = m.group(1).lower()
            found[suffix] = old_name

        # Verificación suave: deben existir 6 ficheros distintos (ang/cld/vel x asc/prj)
        if len(found) != 6:
            incomplete_cases.append((dt_key, f"se encontraron {len(found)} ficheros (esperados 6)"))

        for suffix, old_name in found.items():
            new_name = base + suffix
            plan_rows.append((old_name, new_name, dt_key, suffix))

    plan_df = pd.DataFrame(plan_rows, columns=["old_name", "new_name", "dt_key", "suffix"])


        
    # calcular timestamps únicos encontrados
    timestamps_found = set()
    for name in candidate_names:
        m = re.search(r"\d{2}-\d{2}-\d{4}_\d{4}", name)
        if m:
            timestamps_found.add(m.group(0))

    stats = {
        "n_cases": int(df.shape[0]),
        "n_candidates": int(len(candidate_names)),
        "n_datetimes_found": len(timestamps_found),
        "n_complete_cases": int(df.shape[0] - len(missing_cases)),
        "n_renames": int(plan_df.shape[0]),
        "missing_cases": missing_cases,
        "incomplete_cases": incomplete_cases,
    }
    
    
    
    return plan_df, stats


# ============================================================
# APLICACIÓN DEL PLAN (RECONSTRUYE RUTAS DESDE OUT_DIR)
# ============================================================
def apply_plan_names_only2(plan_df: pd.DataFrame, out_dir: Path, recursive: bool) -> dict:
    """
    Aplica el renombrado usando:
      old_path = (ruta real del fichero antiguo)
      new_path = old_path.with_name(new_name)
    Para resolver old_name->Path, se crea un diccionario name->Path desde el directorio.
    """
    name_to_path = list_candidate_files(out_dir, recursive=recursive)

    missing_files = []
    collisions = []
    applied = 0

    for _, r in plan_df.iterrows():
        old_name = str(r["old_name"])
        new_name = str(r["new_name"])

        old_path = name_to_path.get(old_name)
        if old_path is None or not old_path.exists():
            missing_files.append(old_name)
            continue

        new_path = old_path.with_name(new_name)
        if new_path.exists():
            collisions.append(new_name)
            continue

        old_path.rename(new_path)
        applied += 1

        # actualiza el diccionario para renombres posteriores
        del name_to_path[old_name]
        name_to_path[new_name] = new_path

    return {"applied": applied, "missing_files": missing_files, "collisions": collisions}




def apply_plan_names_only(plan_df: pd.DataFrame, out_dir: Path, dest_dir: Path, recursive: bool) -> dict:
    """
    Aplica el renombrado MOVIENDO los ficheros desde out_dir a dest_dir con el nombre nuevo.
    El plan contiene solo nombres (old_name, new_name).
    """
    dest_dir.mkdir(parents=True, exist_ok=True)

    name_to_path = list_candidate_files(out_dir, recursive=recursive)

    missing_files = []
    collisions = []
    applied = 0

    for _, r in plan_df.iterrows():
        old_name = str(r["old_name"])
        new_name = str(r["new_name"])

        old_path = name_to_path.get(old_name)
        if old_path is None or not old_path.exists():
            missing_files.append(old_name)
            continue

        dest_path = dest_dir / new_name
        if dest_path.exists():
            collisions.append(new_name)
            continue

        # mueve + renombra
        old_path.replace(dest_path)
        applied += 1

        # actualiza diccionario (por si hay dependencias internas)
        del name_to_path[old_name]

    return {"applied": applied, "missing_files": missing_files, "collisions": collisions}




# ============================================================
# FUNCIÓN PRINCIPAL PARA PIPELINES
# ============================================================
def run_rename() -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    plan_df, stats = build_plan_names_only(
        cases_csv=CASES_CSV,
        out_dir=OUT_DIR,
        prefix=PREFIX,
        date_col=DATE_COL,
        dir_col=DIR_COL,
        speed_col=SPEED_COL,
        recursive=RECURSIVE,
    )

    plan_df[["old_name", "new_name"]].to_csv(PLAN_CSV, index=False, encoding="utf-8")

    print(f"[OK] Plan guardado en: {PLAN_CSV}")
    print(f"     Casos CSV:              {stats['n_cases']}")
    print(f"     Ficheros candidatos:    {stats['n_candidates']}")
    print(f"     Date-times encontrados: {stats['n_datetimes_found']}")
    print(f"     Casos completos:        {stats['n_complete_cases']}")
    print(f"     Renombrados previstos:  {stats['n_renames']}")

    if stats["missing_cases"]:
        print(f"[WARN] Casos sin match: {len(stats['missing_cases'])}")
        print("       Ejemplos:", stats["missing_cases"][:10])

    if stats["incomplete_cases"]:
        print(f"[WARN] Casos incompletos: {len(stats['incomplete_cases'])}")
        print("       Ejemplo:", stats["incomplete_cases"][0])

    if APPLY:
        res = apply_plan_names_only(
            plan_df,
            out_dir=OUT_DIR,
            dest_dir=DEST_DIR,
            recursive=RECURSIVE,
        )
        print(f"[OK] Movidos+renombrados: {res['applied']}")
        if res["missing_files"]:
            print(f"[WARN] Missing files: {len(res['missing_files'])}")
        if res["collisions"]:
            print(f"[WARN] Colisiones: {len(res['collisions'])}")

        stats["apply_result"] = res

    # Diagnóstico posterior
    diag_df = build_diagnostics_df(
        cases_csv=CASES_CSV,
        out_dir=OUT_DIR if not APPLY else DEST_DIR,
        recursive=RECURSIVE,
    )
    diag_df.to_csv(DIAG_CSV, index=False, encoding="utf-8")

    out_summary = summarize_directory(OUT_DIR, recursive=True)
    ren_summary = summarize_directory(DEST_DIR, recursive=True) if DEST_DIR.exists() else None

    write_summary_txt(
        path=SUMMARY_TXT,
        stats=stats,
        diag_df=diag_df,
        out_summary=out_summary,
        ren_summary=ren_summary,
    )

    print(f"[OK] Diagnóstico CSV: {DIAG_CSV}")
    print(f"[OK] Resumen TXT:     {SUMMARY_TXT}")

    return plan_df, stats, diag_df


# ============================================================




In [14]:
from pathlib import Path
import re

WN_FILE_RE = re.compile(
    r"^(?P<prefix>.+?)_"
    r"(?P<dt>\d{2}-\d{2}-\d{4}_\d{4})_"
    r"(?P<res>\d+(?:\.\d+)?m)_"
    r"(?P<var>ang|cld|vel)\."
    r"(?P<ext>asc|prj)$",
    re.IGNORECASE
)

def iter_files(out_dir: Path, recursive: bool):
    return out_dir.rglob("*") if recursive else out_dir.glob("*")

def list_candidate_files_index(out_dir: Path, recursive: bool) -> list[Path]:
    files = []
    for p in iter_files(out_dir, recursive):
        if p.is_file() and WN_FILE_RE.match(p.name):
            files.append(p)
    return files

def build_index_by_datetime(out_dir: Path, recursive: bool) -> dict[str, dict[str, Path]]:
    """
    Índice:
    {
        '01-04-2025_1745': {
            '_ang.asc': Path(...),
            '_ang.prj': Path(...),
            '_cld.asc': Path(...),
            '_cld.prj': Path(...),
            '_vel.asc': Path(...),
            '_vel.prj': Path(...),
        },
        ...
    }
    """
    idx: dict[str, dict[str, Path]] = {}

    for p in list_candidate_files_index(out_dir, recursive):
        m = WN_FILE_RE.match(p.name)
        if not m:
            continue

        dt_key = m.group("dt")
        suffix = f"_{m.group('var').lower()}.{m.group('ext').lower()}"

        if dt_key not in idx:
            idx[dt_key] = {}

        idx[dt_key][suffix] = p

    return idx

In [15]:
from collections import Counter
from pathlib import Path
import pandas as pd
import re


def build_diagnostics_df(cases_csv: Path, out_dir: Path, recursive: bool) -> pd.DataFrame:
    """
    Genera un dataframe de diagnóstico por timestamp esperado.
    """
    df = pd.read_csv(cases_csv).copy()

    df["dt_key"]  = df[DATE_COL].map(fmt_datetime)
    df["dir_key"] = df[DIR_COL].map(fmt_dir)
    df["spd_key"] = df[SPEED_COL].map(fmt_speed)

    df["target_base"] = (
        PREFIX + "_" +
        df["dir_key"] + "_" +
        df["spd_key"] + "_" +
        df["dt_key"]
    )

    idx = build_index_by_datetime(out_dir, recursive=recursive)

    rows = []
    for _, row in df.iterrows():
        dt_key = row["dt_key"]
        found = idx.get(dt_key, {})

        found_suffixes = set(found.keys())
        missing_suffixes = sorted(EXPECTED_SUFFIXES - found_suffixes)
        extra_suffixes   = sorted(found_suffixes - EXPECTED_SUFFIXES)

        rows.append({
            "date_time_original": row[DATE_COL],
            "dt_key": dt_key,
            "dir_deg": row[DIR_COL],
            "speed": row[SPEED_COL],
            "target_base": row["target_base"],
            "n_found": len(found_suffixes),
            "is_complete": found_suffixes == EXPECTED_SUFFIXES,
            "is_missing": len(found_suffixes) == 0,
            "missing_suffixes": ";".join(missing_suffixes),
            "extra_suffixes": ";".join(extra_suffixes),
            "found_suffixes": ";".join(sorted(found_suffixes)),
        })

    diag_df = pd.DataFrame(rows)
    return diag_df


def summarize_directory(dir_path: Path, recursive: bool = True) -> dict:
    """
    Resume el contenido real de un directorio.
    """
    files = [p for p in (dir_path.rglob("*") if recursive else dir_path.glob("*")) if p.is_file()]
    ext_counter = Counter(p.suffix.lower() for p in files)

    ts_counter = Counter()
    unmatched = []

    for p in files:
        m = WN_FILE_RE.match(p.name)
        if m:
            ts_counter[m.group("dt")] += 1
        else:
            unmatched.append(p.name)

    return {
        "path": str(dir_path.resolve()),
        "n_files": len(files),
        "ext_counter": dict(ext_counter),
        "n_timestamps": len(ts_counter),
        "timestamp_counter_distribution": dict(Counter(ts_counter.values())),
        "unmatched_examples": unmatched[:20],
    }


def write_summary_txt(path: Path, stats: dict, diag_df: pd.DataFrame, out_summary: dict, ren_summary: dict | None = None):
    """
    Escribe un resumen legible en texto.
    """
    lines = []
    lines.append("RENAME SUMMARY")
    lines.append("=" * 80)
    lines.append("")
    lines.append("PLAN")
    lines.append(f"n_cases               : {stats.get('n_cases')}")
    lines.append(f"n_candidates          : {stats.get('n_candidates')}")
    lines.append(f"n_datetimes_found     : {stats.get('n_datetimes_found')}")
    lines.append(f"n_complete_cases      : {stats.get('n_complete_cases')}")
    lines.append(f"n_renames             : {stats.get('n_renames')}")
    lines.append("")

    if "apply_result" in stats:
        ar = stats["apply_result"]
        lines.append("APPLY RESULT")
        lines.append(f"applied               : {ar.get('applied')}")
        lines.append(f"missing_files         : {len(ar.get('missing_files', []))}")
        lines.append(f"collisions            : {len(ar.get('collisions', []))}")
        lines.append("")

    lines.append("DIAGNOSTICS")
    lines.append(f"rows                  : {len(diag_df)}")
    lines.append(f"complete              : {int(diag_df['is_complete'].sum())}")
    lines.append(f"missing               : {int(diag_df['is_missing'].sum())}")
    lines.append(f"incomplete            : {int((~diag_df['is_complete'] & ~diag_df['is_missing']).sum())}")
    lines.append("")

    lines.append("OUT_WN")
    lines.append(f"path                  : {out_summary['path']}")
    lines.append(f"n_files               : {out_summary['n_files']}")
    lines.append(f"n_timestamps          : {out_summary['n_timestamps']}")
    lines.append(f"ext_counter           : {out_summary['ext_counter']}")
    lines.append(f"timestamp_distribution: {out_summary['timestamp_counter_distribution']}")
    if out_summary["unmatched_examples"]:
        lines.append(f"unmatched_examples    : {out_summary['unmatched_examples']}")
    lines.append("")

    if ren_summary is not None:
        lines.append("OUT_WN_REN")
        lines.append(f"path                  : {ren_summary['path']}")
        lines.append(f"n_files               : {ren_summary['n_files']}")
        lines.append(f"n_timestamps          : {ren_summary['n_timestamps']}")
        lines.append(f"ext_counter           : {ren_summary['ext_counter']}")
        lines.append(f"timestamp_distribution: {ren_summary['timestamp_counter_distribution']}")
        if ren_summary["unmatched_examples"]:
            lines.append(f"unmatched_examples    : {ren_summary['unmatched_examples']}")
        lines.append("")

    missing_rows = diag_df[diag_df["is_missing"]]
    if not missing_rows.empty:
        lines.append("MISSING CASES (first 20)")
        for _, r in missing_rows.head(20).iterrows():
            lines.append(f"  {r['dt_key']}")
        lines.append("")

    incomplete_rows = diag_df[(~diag_df["is_complete"]) & (~diag_df["is_missing"])]
    if not incomplete_rows.empty:
        lines.append("INCOMPLETE CASES (first 20)")
        for _, r in incomplete_rows.head(20).iterrows():
            lines.append(f"  {r['dt_key']} -> found={r['found_suffixes']} missing={r['missing_suffixes']}")
        lines.append("")

    path.write_text("\n".join(lines), encoding="utf-8")

In [16]:
APPLY = True
plan_df, stats, diag_df = run_rename()

print("Resumen del plan de renombrado:")
stats
diag_df.head()
diag_df[~diag_df["is_complete"]]


[OK] Plan guardado en: E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\rename_plan.csv
     Casos CSV:              360
     Ficheros candidatos:    2160
     Date-times encontrados: 360
     Casos completos:        360
     Renombrados previstos:  2160
[OK] Movidos+renombrados: 2160
[OK] Diagnóstico CSV: E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\rename_diagnostics.csv
[OK] Resumen TXT:     E:\mario\trabajos2\viesgo_edp_portugal_cic\estudios_microclimaticos\Corredoria_Grado_1_y_2\rename_summary.txt
Resumen del plan de renombrado:


,date_time_original,dt_key,dir_deg,speed,target_base,n_found,is_complete,is_missing,missing_suffixes,extra_suffixes,found_suffixes
0,2025-01-01T00:00:00Z,01-01-2025_0000,0.0,1.0,Corredoria_Grado_1_y_2_000_0_01_0_01-01-2025_0000,0,False,True,_ang.asc;_ang.prj;_cld.asc;_cld.prj;_vel.asc;_...,,
1,2025-01-01T00:15:00Z,01-01-2025_0015,1.0,1.0,Corredoria_Grado_1_y_2_001_0_01_0_01-01-2025_0015,0,False,True,_ang.asc;_ang.prj;_cld.asc;_cld.prj;_vel.asc;_...,,
2,2025-01-01T00:30:00Z,01-01-2025_0030,2.0,1.0,Corredoria_Grado_1_y_2_002_0_01_0_01-01-2025_0030,0,False,True,_ang.asc;_ang.prj;_cld.asc;_cld.prj;_vel.asc;_...,,
3,2025-01-01T00:45:00Z,01-01-2025_0045,3.0,1.0,Corredoria_Grado_1_y_2_003_0_01_0_01-01-2025_0045,0,False,True,_ang.asc;_ang.prj;_cld.asc;_cld.prj;_vel.asc;_...,,
4,2025-01-01T01:00:00Z,01-01-2025_0100,4.0,1.0,Corredoria_Grado_1_y_2_004_0_01_0_01-01-2025_0100,0,False,True,_ang.asc;_ang.prj;_cld.asc;_cld.prj;_vel.asc;_...,,
...,...,...,...,...,...,...,...,...,...,...,...
355,2025-01-04T16:45:00Z,01-04-2025_1645,355.0,1.0,Corredoria_Grado_1_y_2_355_0_01_0_01-04-2025_1645,0,False,True,_ang.asc;_ang.prj;_cld.asc;_cld.prj;_vel.asc;_...,,
356,2025-01-04T17:00:00Z,01-04-2025_1700,356.0,1.0,Corredoria_Grado_1_y_2_356_0_01_0_01-04-2025_1700,0,False,True,_ang.asc;_ang.prj;_cld.asc;_cld.prj;_vel.asc;_...,,
357,2025-01-04T17:15:00Z,01-04-2025_1715,357.0,1.0,Corredoria_Grado_1_y_2_357_0_01_0_01-04-2025_1715,0,False,True,_ang.asc;_ang.prj;_cld.asc;_cld.prj;_vel.asc;_...,,
358,2025-01-04T17:30:00Z,01-04-2025_1730,358.0,1.0,Corredoria_Grado_1_y_2_358_0_01_0_01-04-2025_1730,0,False,True,_ang.asc;_ang.prj;_cld.asc;_cld.prj;_vel.asc;_...,,
